# scE2TM: Basic Usage Tutorial

This tutorial provides a step-by-step guide to running scE2TM on a sample single-cell dataset (Wang et al.).

## Dataset

We use the **Wang** dataset as an example, which includes:
- Gene expression matrix (`Wang_HIGHPRE.csv`)
- Cell type annotations (`Wang_cell_anno.csv`)
- Foundation model embeddings (`Wang.csv`)

All data files are located in the `./data/` directory.

## Before You Start

Make sure you have installed scE2TM:
```bash
pip install scE2TM

In [1]:
import scE2TM
print(f"scE2TM path: {scE2TM.__file__}")

scE2TM path: /mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/scE2TM/__init__.py


In [2]:
import random
import torch
import numpy as np
import yaml
import os
from pathlib import Path

from scE2TM.runners.Runner import Runner
from scE2TM.utils.data.SingleCellDataHandler import SingleCellDataHandler
from scE2TM.utils.data import file_utils

In [3]:
def set_seed(seed):
    """Set random seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def setup_device(gpu_id):
    """
    Setup computation device
    
    Args:
        gpu_id: GPU ID, -1 for CPU
    Returns:
        device: torch.device object
    """
    if gpu_id >= 0 and torch.cuda.is_available():
        device = torch.device(f'cuda:{gpu_id}')
        print(f"Using GPU: {torch.cuda.get_device_name(gpu_id)}")
    else:
        device = torch.device('cpu')
        print("Using CPU")
    return device

In [4]:
class Args:
    pass

args = Args()

args.model = 'scE2TM'
args.config = './configs/scE2TM.yaml'  
args.num_topics = 100
args.num_neighbors = 15
args.dataset_name = 'Wang'  
args.num_top_genes = 10
args.tac_weight = 1.0
args.gpu_id = 0
args.data_dir = './data'
args.output_dir = './output'


In [5]:
file_utils.update_args(args, path=args.config)

print(f"batch_size: {args.batch_size}")
print(f"learning_rate: {args.learning_rate}")
print(f"epochs: {args.epochs}")
print(f"beta_temp: {args.beta_temp}")
print(f"en1_units: {args.en1_units}")
print(f"dropout: {args.dropout}")
print(f"weight_loss_ECR: {args.weight_loss_ECR}")

batch_size: 512
learning_rate: 0.001
epochs: 500
beta_temp: 0.2
en1_units: 200
dropout: 0
weight_loss_ECR: 100


In [6]:
GLOBAL_SEED = 1
set_seed(GLOBAL_SEED)

device = setup_device(args.gpu_id)
args.device = device

print(f"Using device: {device}")

Using GPU: NVIDIA GeForce GTX 1080 Ti
Using device: cuda:0


In [7]:
data_handler = SingleCellDataHandler(
    dataset_name=args.dataset_name,
    batch_size=args.batch_size,
    n_neighbors=args.num_neighbors,
    data_dir=args.data_dir,
    output_dir=args.output_dir,
    device=device
)

Expression data shape: (457, 5000)
Foundation embeddings shape: torch.Size([457, 512])
Number of neighbors: 15
Wed Mar 25 08:57:45 2026 Finding Nearest Neighbors
Wed Mar 25 08:57:45 2026 Building RP forest with 6 trees
Wed Mar 25 08:57:56 2026 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	 3  /  9
	 4  /  9
	Stopping threshold met -- exiting after 4 iterations
Wed Mar 25 08:58:27 2026 Finished Nearest Neighbor Search
Number of connected components: 338
	completed  0  /  457 epochs
	completed  45  /  457 epochs
	completed  90  /  457 epochs
	completed  135  /  457 epochs
	completed  180  /  457 epochs
	completed  225  /  457 epochs
	completed  270  /  457 epochs
	completed  315  /  457 epochs
	completed  360  /  457 epochs
	completed  405  /  457 epochs
	completed  450  /  457 epochs
Wed Mar 25 08:58:27 2026 Finding Nearest Neighbors
Wed Mar 25 08:58:27 2026 Building RP forest with 6 trees
Wed Mar 25 08:58:28 2026 NN descent for 9 iterations
	 1  /  9
	 2  /  9
	 3  /  9
	Stopping th

In [8]:
args.vocab_size = data_handler.expression_data.shape[1]
args.gene_embeddings = data_handler.gene_embeddings

print(f"args.vocab_size = {args.vocab_size}")
print(f"args.en1_units = {args.en1_units}") 

args.vocab_size = 5000
args.en1_units = 200


In [9]:
runner = Runner(args)

In [10]:
beta = runner.train(
    train_loader=data_handler.train_loader,
    test_loader=data_handler.test_loader,
    gene_vocab=data_handler.gene_vocab,
    num_top_genes=args.num_top_genes,
    cell_labels=data_handler.test_labels
)

===>Warning: use lr_scheduler
Adjusting learning rate of group 0 to 1.0000e-03.
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 001 loss: 35809.566 topic_model_loss: 35669.078 cross_modal_loss: -30.103 ecr_loss: 170.591
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 002 loss: 34293.457 topic_model_loss: 34153.953 cross_modal_loss: -30.151 ecr_loss: 169.655
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 003 loss: 33817.156 topic_model_loss: 33679.438 cross_modal_loss: -30.359 ecr_loss: 168.077
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 004 loss: 33421.453 topic_model_loss: 33283.898 cross_modal_loss: -29.643 ecr_loss: 167.198
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 005 loss: 33296.590 topic_model_loss: 33159.469 cross_modal_loss: -29.494 ecr_loss: 166.612
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 006 loss: 33261.637 topic_model_loss: 33125.598 cross_modal_loss: -29.763 ecr_loss: 165.801
Adjusting learning rate of

/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 score: f1_macro = 0.64775, f1_micro = 0.94203
precision score: precision_macro = 0.62361, precision_micro = 0.94203
recall score: recall_macro = 0.6807, recall_micro = 0.94203
scE2TM Purity: 0.8796
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 101 loss: 32146.539 topic_model_loss: 32063.266 cross_modal_loss: -32.082 ecr_loss: 115.356
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 102 loss: 32134.756 topic_model_loss: 32051.773 cross_modal_loss: -32.105 ecr_loss: 115.088
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 103 loss: 32143.068 topic_model_loss: 32060.291 cross_modal_loss: -32.119 ecr_loss: 114.897
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 104 loss: 32179.250 topic_model_loss: 32097.229 cross_modal_loss: -32.106 ecr_loss: 114.128
Adjusting learning rate of group 0 to 1.0000e-03.
Epoch: 105 loss: 32137.615 topic_model_loss: 32055.326 cross_modal_loss: -32.138 ecr_loss: 114.427
Adjusting learning rate of group 0 to 1.0000e-03.
Epoc

/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 201 loss: 31818.611 topic_model_loss: 31756.217 cross_modal_loss: -32.581 ecr_loss: 94.977
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 202 loss: 31826.248 topic_model_loss: 31764.016 cross_modal_loss: -32.573 ecr_loss: 94.806
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 203 loss: 31815.447 topic_model_loss: 31753.199 cross_modal_loss: -32.536 ecr_loss: 94.782
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 204 loss: 31806.695 topic_model_loss: 31744.775 cross_modal_loss: -32.642 ecr_loss: 94.562
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 205 loss: 31808.863 topic_model_loss: 31747.104 cross_modal_loss: -32.552 ecr_loss: 94.311
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 206 loss: 31812.346 topic_model_loss: 31750.664 cross_modal_loss: -32.536 ecr_loss: 94.219
Adjusting learning rate of group 0 to 5.0000e-04.
Epoch: 207 loss: 31804.990 topic_model_loss: 31743.465 cross_m

/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 301 loss: 31626.402 topic_model_loss: 31574.641 cross_modal_loss: -32.973 ecr_loss: 84.734
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 302 loss: 31630.719 topic_model_loss: 31579.002 cross_modal_loss: -32.926 ecr_loss: 84.641
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 303 loss: 31626.088 topic_model_loss: 31574.381 cross_modal_loss: -32.927 ecr_loss: 84.633
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 304 loss: 31624.555 topic_model_loss: 31573.039 cross_modal_loss: -33.012 ecr_loss: 84.525
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 305 loss: 31632.201 topic_model_loss: 31580.668 cross_modal_loss: -32.983 ecr_loss: 84.516
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 306 loss: 31619.941 topic_model_loss: 31568.432 cross_modal_loss: -32.958 ecr_loss: 84.467
Adjusting learning rate of group 0 to 2.5000e-04.
Epoch: 307 loss: 31623.033 topic_model_loss: 31571.629 cross_m

/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 401 loss: 31513.490 topic_model_loss: 31467.301 cross_modal_loss: -33.236 ecr_loss: 79.426
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 402 loss: 31524.791 topic_model_loss: 31478.564 cross_modal_loss: -33.185 ecr_loss: 79.411
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 403 loss: 31516.672 topic_model_loss: 31470.475 cross_modal_loss: -33.211 ecr_loss: 79.408
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 404 loss: 31516.689 topic_model_loss: 31470.512 cross_modal_loss: -33.203 ecr_loss: 79.382
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 405 loss: 31513.883 topic_model_loss: 31467.727 cross_modal_loss: -33.187 ecr_loss: 79.342
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 406 loss: 31512.449 topic_model_loss: 31466.301 cross_modal_loss: -33.165 ecr_loss: 79.315
Adjusting learning rate of group 0 to 1.2500e-04.
Epoch: 407 loss: 31516.123 topic_model_loss: 31469.986 cross_m

/mnt/rao/home/chenhg/anaconda3/envs/scE2TM_test/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
